# Student-t Oracle DML 基准：$p=5$ 与 $p=10$

本 notebook 用真实 nuisance functions 构造 Oracle DML，诊断可行 DML 的低覆盖率是否主要来自第一阶段机器学习误差。参数与现有 Student-t notebooks 保持一致：$N=1000$、$\theta_0=1$、1000 次 Monte Carlo 重复、相同自由度网格和正态基准。

部分线性模型为
$$Y=D\theta_0+g_0(X)+U,\qquad D=m_0(X)+V.$$
Oracle 已知
$$m_0(X)=E[D\mid X],\qquad \ell_0(X)=E[Y\mid X]=\theta_0m_0(X)+g_0(X),$$
因此精确残差为
$$\widetilde D=D-m_0(X)=V,\qquad \widetilde Y=Y-\ell_0(X)=\theta_0V+U.$$

Student-t 和 Gaussian baseline 都关于 0 对称，所以这里的条件均值与条件中位数 nuisance functions 相同。Oracle 已知真实函数，不存在训练样本泄漏，也无需 cross-fitting。

In [ ]:
# ============================================================
# 调参区：公共参数与现有 Student-t notebooks 保持一致
# ============================================================
P_GRID = [5, 10]
N_SIGNAL = 5
N = 1000
THETA_0 = 1.0
T_DF_GRID = [2.1, 2.3, 2.6, 3, 5, 10, 20]
NORMAL_BASELINE = "N(0,1)"
ERROR_GRID = [*T_DF_GRID, NORMAL_BASELINE]
DGP_NAMES = ["DGP1_linear", "DGP2_nonlinear"]

N_FOLDS = 5  # 仅为记录与可行 DML 一致；Oracle 不需要交叉拟合
N_REPLICATIONS = 1000
BASE_SEED = 20260826
N_JOBS = -1

QUICK_TEST = False
QUICK_N = 300
QUICK_REPLICATIONS = 2

SAVE_RESULTS = True
OUTPUT_CSV = None  # 导入依赖后自动设为桌面 CSV

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import norm

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

if OUTPUT_CSV is None:
    OUTPUT_CSV = Path.home() / "Desktop" / "dml_oracle_student_t_p5_p10_summary.csv"
else:
    OUTPUT_CSV = Path(OUTPUT_CSV)

assert P_GRID == [5, 10]
assert N_SIGNAL == 5
assert all(p >= N_SIGNAL for p in P_GRID)
assert all(df > 2 for df in T_DF_GRID)
print(f"参数检查完成。正式结果将保存到：{OUTPUT_CSV}")

## 1. 真实 DGP 与 Oracle 残差检查

`p=10` 时仍只有前 5 维进入真实函数，$X_6,\ldots,X_{10}$ 的结构作用为 0。下面完整生成一次数据并验证 Oracle 残差确实等于 $V$ 和 $\theta_0V+U$。正式 Monte Carlo 随后直接使用这个代数等价的残差表示，以避免重复生成会被精确消去的 $X$。

In [ ]:
def ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])


def nuisance_functions(X: np.ndarray, dgp_name: str):
    x1, x2, x3, x4, x5 = X[:, :N_SIGNAL].T
    if dgp_name == "DGP1_linear":
        m0 = 0.80*x1 - 0.50*x2 + 0.30*x3 + 0.20*x4 - 0.10*x5
        g0 = 1.00*x1 + 0.50*x2 - 0.50*x3 + 0.25*x4 + 0.10*x5
    elif dgp_name == "DGP2_nonlinear":
        m0 = (0.60*np.sin(x1) + 0.35*(x2**2 - 1.0)
              + 0.25*x3*x4 - 0.20*np.tanh(x5))
        g0 = (1.20*np.sin(x1) + 0.70*(x2**2 - 1.0)
              + 0.50*x3*x4 + 0.30*np.tanh(x5))
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def standardized_student_t(rng: np.random.Generator, df: float, size: int):
    return rng.standard_t(df, size=size) * np.sqrt((df - 2.0) / df)


def draw_u(rng: np.random.Generator, error_case, size: int):
    if error_case == NORMAL_BASELINE:
        return rng.normal(0.0, 1.0, size=size)
    return standardized_student_t(rng, float(error_case), size)


def generate_full_sample(n: int, p: int, error_case, dgp_name: str, seed: int):
    rng = np.random.default_rng(seed)
    X = rng.multivariate_normal(np.zeros(p), ar1_covariance(p), size=n)
    m0, g0 = nuisance_functions(X, dgp_name)
    V = rng.normal(0.0, 1.0, size=n)
    U = draw_u(rng, error_case, n)
    D = m0 + V
    Y = THETA_0*D + g0 + U
    ell0 = THETA_0*m0 + g0
    return X, D, Y, U, V, m0, ell0


for p_demo in P_GRID:
    X, D, Y, U, V, m0, ell0 = generate_full_sample(
        n=200, p=p_demo, error_case=NORMAL_BASELINE,
        dgp_name="DGP2_nonlinear", seed=BASE_SEED + p_demo,
    )
    np.testing.assert_allclose(D - m0, V, atol=1e-12)
    np.testing.assert_allclose(Y - ell0, THETA_0*V + U, atol=1e-12)
    print(f"p={p_demo}: X shape={X.shape}，Oracle 残差恒等式检查通过。")

## 2. Oracle Mean DML 与 Oracle Median DML

第二阶段估计式和原 Student-t notebooks 保持一致。区别仅在于这里直接使用真实残差，不训练 `HistGradientBoostingRegressor`。

In [ ]:
def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cutoff = 0.5 * weights_sorted.sum()
    index = np.searchsorted(np.cumsum(weights_sorted), cutoff, side="left")
    return float(values_sorted[min(index, len(values_sorted)-1)])


def fit_mean_oracle(d_res: np.ndarray, y_res: np.ndarray):
    jacobian = np.mean(d_res**2)
    if jacobian <= 1e-12:
        return np.nan, np.nan
    theta_hat = np.mean(d_res*y_res) / jacobian
    score = d_res * (y_res - theta_hat*d_res)
    influence = score / jacobian
    se_hat = np.sqrt(np.var(influence, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)


def fit_median_oracle(d_res: np.ndarray, y_res: np.ndarray):
    keep = np.abs(d_res) > 1e-10
    if not np.any(keep):
        return np.nan, np.nan
    ratios = y_res[keep] / d_res[keep]
    weights = np.abs(d_res[keep])
    theta_hat = weighted_median(ratios, weights)
    residual = y_res - theta_hat*d_res
    n = len(residual)
    mad = np.median(np.abs(residual - np.median(residual))) / 0.67448975
    robust_scale = max(float(mad), 1e-6)
    bandwidth = max(1.06*robust_scale*n**(-1.0/5.0), 1e-6)
    kernel_at_zero = norm.pdf(residual / bandwidth) / bandwidth
    A_hat = np.mean((d_res**2) * kernel_at_zero)
    B_hat = 0.25 * np.mean(d_res**2)
    se_hat = np.sqrt(B_hat / (A_hat**2*n)) if A_hat > 1e-12 else np.nan
    return float(theta_hat), float(se_hat)


def estimate_oracle(V: np.ndarray, U: np.ndarray):
    d_res = V
    y_res = THETA_0*V + U
    return [
        ("Oracle_Mean_DML", *fit_mean_oracle(d_res, y_res)),
        ("Oracle_Median_DML", *fit_median_oracle(d_res, y_res)),
    ]

In [ ]:
def one_replication(error_case, replication: int, n_use: int):
    error_id = ERROR_GRID.index(error_case)
    # p 与 DGP 共用同一组 V、U，实现严格配对；Oracle 理论上不受它们影响。
    seed = BASE_SEED + 1_000*error_id + replication
    rng = np.random.default_rng(seed)
    V = rng.normal(0.0, 1.0, size=n_use)
    U = draw_u(rng, error_case, n_use)
    estimates = estimate_oracle(V, U)

    rows = []
    for p in P_GRID:
        for dgp_name in DGP_NAMES:
            for estimator, theta_hat, se_hat in estimates:
                ci_low = theta_hat - 1.96*se_hat
                ci_high = theta_hat + 1.96*se_hat
                rows.append({
                    "dgp": dgp_name,
                    "p": p,
                    "n": n_use,
                    "error_distribution": (NORMAL_BASELINE if error_case == NORMAL_BASELINE
                                           else f"t({error_case:g})"),
                    "tail_order": error_id,
                    "t_df": np.nan if error_case == NORMAL_BASELINE else float(error_case),
                    "replication": replication,
                    "estimator": estimator,
                    "theta_hat": theta_hat,
                    "se_hat": se_hat,
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                    "covered": bool(ci_low <= THETA_0 <= ci_high),
                    "sample_u_variance": np.var(U, ddof=1),
                    "sample_max_abs_u": np.max(np.abs(U)),
                })
    return rows


def run_simulation():
    n_use = QUICK_N if QUICK_TEST else N
    r_use = QUICK_REPLICATIONS if QUICK_TEST else N_REPLICATIONS
    tasks = [
        (error_case, replication, n_use)
        for error_case in ERROR_GRID
        for replication in range(r_use)
    ]
    mode = "QUICK_TEST" if QUICK_TEST else "FORMAL"
    print(f"开始 [{mode}]：p={P_GRID}, n={n_use}, errors={ERROR_GRID}, repetitions={r_use}")
    start = time.perf_counter()
    nested_rows = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(one_replication)(*task) for task in tasks
    )
    rows = [row for block in nested_rows for row in block]
    print(f"完成，用时 {time.perf_counter()-start:.1f} 秒，共 {len(rows)} 条结果。")
    return pd.DataFrame(rows)


results = run_simulation()
results.head()

In [ ]:
def summarize_simulation(results_df: pd.DataFrame):
    work = results_df.copy()
    work["error"] = work["theta_hat"] - THETA_0
    work["squared_error"] = work["error"]**2
    summary = (
        work.groupby(
            ["p", "dgp", "error_distribution", "tail_order", "estimator"],
            as_index=False,
        )
        .agg(
            t_df=("t_df", "first"),
            mean_estimate=("theta_hat", "mean"),
            bias=("error", "mean"),
            sd=("theta_hat", "std"),
            mean_se=("se_hat", "mean"),
            mse=("squared_error", "mean"),
            coverage=("covered", "mean"),
            mean_sample_u_variance=("sample_u_variance", "mean"),
            mean_sample_max_abs_u=("sample_max_abs_u", "mean"),
            valid_runs=("theta_hat", "count"),
        )
    )
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    return summary.sort_values(["p", "dgp", "tail_order", "estimator"]).reset_index(drop=True)


summary = summarize_simulation(results)
display(summary.round({
    "mean_estimate": 4, "bias": 4, "sd": 4, "mean_se": 4,
    "rmse": 4, "coverage": 3, "mean_sample_u_variance": 3,
    "mean_sample_max_abs_u": 2,
}))

gaussian_check = summary[summary["error_distribution"] == NORMAL_BASELINE]
print("Gaussian baseline Oracle 结果：")
display(gaussian_check[[
    "p", "dgp", "estimator", "bias", "sd", "mean_se", "coverage"
]].round(4))

if SAVE_RESULTS and not QUICK_TEST:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"正式汇总结果已保存：{OUTPUT_CSV.resolve()}")
elif QUICK_TEST:
    print("QUICK_TEST=True：不写出测试 CSV。")
else:
    print("SAVE_RESULTS=False：未写出 CSV。")

In [ ]:
# p 与 DGP 在 Oracle 下使用严格配对的相同残差，因此曲线理论上重合。
plot_data = summary[(summary["p"] == 5) & (summary["dgp"] == "DGP1_linear")]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (metric, label) in zip(
    axes, [("bias", "Bias"), ("rmse", "RMSE"), ("coverage", "95% CI coverage")]
):
    sns.lineplot(data=plot_data, x="tail_order", y=metric, hue="estimator", markers=True, ax=ax)
    ax.set_xticks(range(len(ERROR_GRID)))
    ax.set_xticklabels(
        [f"t({x:g})" if x != NORMAL_BASELINE else NORMAL_BASELINE for x in ERROR_GRID],
        rotation=30,
    )
    ax.set_xlabel("Error distribution")
    ax.set_ylabel(label)
    if metric == "bias":
        ax.axhline(0.0, color="black", linewidth=1, linestyle=":")
    if metric == "coverage":
        ax.axhline(0.95, color="black", linewidth=1, linestyle=":")
        ax.set_ylim(0.0, 1.02)
plt.tight_layout()
plt.show()

## 解释与使用说明

1. Oracle 结果不受 DGP 复杂度或协变量维度影响，因为真实 nuisance functions 被精确消去；代码使用相同的 $V,U$ 实现严格配对，因此不同 `p` 和 DGP 的汇总值应完全一致。
2. 如果 Gaussian Oracle coverage 接近 0.95，而可行 DML 明显偏低，则主要问题来自第一阶段 nuisance estimation，而不是 Gaussian 误差生成或 coverage 计算。
3. 若 Oracle Median DML 的 coverage 仍偏离 0.95，应进一步检查核密度带宽和 sandwich 标准误；它不涉及第一阶段机器学习误差。
4. 调试时设置 `QUICK_TEST=True`；正式版本保持 `QUICK_TEST=False`，运行后自动把汇总结果保存到桌面的 `dml_oracle_student_t_p5_p10_summary.csv`。